# Exploratory Data Analysis: Support Tickets

This notebook explores the Support Ticket Dataset to justify modeling choices, analyze data distributions, handle missing values, and extract common n-grams.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import itertools
from sklearn.feature_extraction.text import CountVectorizer

%matplotlib inline
sns.set_theme(style="whitegrid")

# Load data
df = pd.read_csv('../tickets.csv')
display(df.head())

## 1. Missing Values Analysis
Checking for null values in critical columns.

In [ ]:
missing_vals = df.isnull().sum()
print("Missing values per column:")
print(missing_vals[missing_vals > 0])

plt.figure(figsize=(10, 6))
sns.heatmap(df.isnull(), yticklabels=False, cbar=False, cmap='viridis')
plt.title('Missing Values Heatmap')
plt.show()

## 2. Class Distribution
Checking if our categories are imbalanced, which affects whether we use simple accuracy or weighted F1 macro scores.

In [ ]:
cat_col = 'Ticket Type' if 'Ticket Type' in df.columns else 'Category'
if cat_col not in df.columns and 'category' in df.columns:
    cat_col = 'category'

if cat_col in df.columns:
    plt.figure(figsize=(12, 6))
    order = df[cat_col].value_counts().index
    sns.countplot(data=df, y=cat_col, order=order, palette="viridis")
    plt.title(f'Distribution of {cat_col}')
    plt.xlabel('Count')
    plt.ylabel('Category')
    plt.show()
    
    print("Class representation:\n", df[cat_col].value_counts(normalize=True))

## 3. Text Length Analysis
Analyzing the character and word counts of Ticket Subjects and Descriptions.

In [ ]:
subj_col = 'Ticket Subject' if 'Ticket Subject' in df.columns else 'subject'
body_col = 'Ticket Description' if 'Ticket Description' in df.columns else 'body'

# Create word count features
if subj_col in df.columns:
    df['subj_word_count'] = df[subj_col].astype(str).apply(lambda x: len(x.split()))
if body_col in df.columns:
    df['body_word_count'] = df[body_col].astype(str).apply(lambda x: len(x.split()))

fig, ax = plt.subplots(1, 2, figsize=(16, 5))
if subj_col in df.columns:
    sns.histplot(df['subj_word_count'], bins=50, ax=ax[0], kde=True)
    ax[0].set_title('Subject Word Count Distribution')
    ax[0].set_xlim(0, df['subj_word_count'].quantile(0.99))

if body_col in df.columns:
    sns.histplot(df['body_word_count'], bins=50, ax=ax[1], kde=True)
    ax[1].set_title('Body Word Count Distribution')
    ax[1].set_xlim(0, df['body_word_count'].quantile(0.99))

plt.show()

## 4. Top N-Grams Extraction
Exploring common unigrams and bigrams to inform our text vectorization strategies.

In [ ]:
def get_top_ngrams(corpus, n=None, ngram_range=(1,1)):
    vec = CountVectorizer(ngram_range=ngram_range, stop_words='english').fit(corpus)
    bag_of_words = vec.transform(corpus)
    sum_words = bag_of_words.sum(axis=0)
    words_freq = [(word, sum_words[0, idx]) for word, idx in vec.vocabulary_.items()]
    words_freq = sorted(words_freq, key=lambda x: x[1], reverse=True)
    return words_freq[:n]

if body_col in df.columns:
    text_sample = df[body_col].dropna().astype(str).sample(min(10000, len(df)), random_state=42)
    
    top_unigrams = get_top_ngrams(text_sample, n=15, ngram_range=(1,1))
    top_bigrams = get_top_ngrams(text_sample, n=15, ngram_range=(2,2))
    
    fig, ax = plt.subplots(1, 2, figsize=(18, 6))
    
    # Unigrams
    udf = pd.DataFrame(top_unigrams, columns=['Unigram', 'Frequency'])
    sns.barplot(x='Frequency', y='Unigram', data=udf, ax=ax[0], palette="Blues_r")
    ax[0].set_title('Top 15 Unigrams')
    
    # Bigrams
    bdf = pd.DataFrame(top_bigrams, columns=['Bigram', 'Frequency'])
    sns.barplot(x='Frequency', y='Bigram', data=bdf, ax=ax[1], palette="Greens_r")
    ax[1].set_title('Top 15 Bigrams')
    
    plt.tight_layout()
    plt.show()